# Aligning two coronal sections of adult mouse brain from MERFISH (squidpy + moscot)

In this notebook, we align two single cell resolution spatial transcriptomics datasets of coronal sections of the adult mouse brain from matched locations with respect to bregma assayed by MERFISH.

The original notebook used STalign affine-only alignment with manually picked landmark points. Here we use `squidpy` with the `moscot` backend for point-to-point alignment via optimal transport, which does not require manual landmarks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load source data

In [ ]:
# Single cell data 1
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz'
df1 = pd.read_csv(fname)
print(df1.head())

In [ ]:
# Create AnnData for source
coords_source = np.column_stack([df1['center_x'].values, df1['center_y'].values])
adata_source = ad.AnnData(
    X=np.zeros((len(coords_source), 1)),
    obs=df1,
)
adata_source.obsm['spatial'] = coords_source

# Plot
fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='source')
ax.legend(markerscale=10)
ax.set_aspect('equal')

## Load target data

In [ ]:
# Single cell data 2
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate2_cell_metadata_S2R2.csv.gz'
df2 = pd.read_csv(fname)

coords_target = np.column_stack([df2['center_x'].values, df2['center_y'].values])
adata_target = ad.AnnData(
    X=np.zeros((len(coords_target), 1)),
    obs=df2,
)
adata_target.obsm['spatial'] = coords_target

# Plot
fig, ax = plt.subplots()
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.2, c='#ff7f0e', label='target')
ax.legend(markerscale=10)
ax.set_aspect('equal')

In [ ]:
# Plot both before alignment
fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='source')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='target')
ax.legend(markerscale=10)
ax.set_aspect('equal')

## Align using moscot (optimal transport)

Unlike the STalign affine-only approach which requires manual landmark selection, moscot finds an optimal transport plan between the two point clouds automatically.

In [ ]:
# Align source to target using moscot
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

## Visualize results

In [ ]:
# Plot results
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1],
           s=1, alpha=0.1, label='source')
ax.scatter(aligned[:, 0], aligned[:, 1],
           s=1, alpha=0.1, label='source aligned')
ax.scatter(coords_target[:, 0], coords_target[:, 1],
           s=1, alpha=0.1, label='target')
ax.legend(markerscale=10)
ax.set_aspect('equal')

In [ ]:
# 2x2 panel summary
fig, ax = plt.subplots(2, 2, figsize=(16, 14))

ax[0][0].scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.1, label='source cells')
ax[0][0].set_title('Source')
ax[0][0].set_aspect('equal')
ax[0][0].legend(markerscale=10)

ax[0][1].scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, c='orange', label='target cells')
ax[0][1].set_title('Target')
ax[0][1].set_aspect('equal')
ax[0][1].legend(markerscale=10)

ax[1][0].scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='aligned source cells')
ax[1][0].set_title('Aligned Source')
ax[1][0].set_aspect('equal')
ax[1][0].legend(markerscale=10)

ax[1][1].scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, c='orange', label='target')
ax[1][1].scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='aligned source')
ax[1][1].set_title('Overlay')
ax[1][1].set_aspect('equal')
ax[1][1].legend(markerscale=10)

## Save results

In [ ]:
# Save aligned coordinates
df_aligned = pd.DataFrame({
    'aligned_x': aligned[:, 0],
    'aligned_y': aligned[:, 1],
})
results = pd.concat([df1, df_aligned], axis=1)
results.head()